## Setup Spark Session

In [36]:

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab01_Enhanced") \
    .getOrCreate()


## Create DataFrame from Dictionary

In [37]:

data = [
    {"id": 1, "name": "Ali", "age": 25, "salary": 4000},
    {"id": 2, "name": "Sara", "age": 30, "salary": 7000},
    {"id": 3, "name": "Omar", "age": 22, "salary": None},
    {"id": 4, "name": "Mona", "age": None, "salary": 5000}
]

df = spark.createDataFrame(data)

df.show()
df.printSchema()


+----+---+----+------+
| age| id|name|salary|
+----+---+----+------+
|  25|  1| Ali|  4000|
|  30|  2|Sara|  7000|
|  22|  3|Omar|  null|
|null|  4|Mona|  5000|
+----+---+----+------+

root
 |-- age: long (nullable = true)
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)




## 🔍 Debugging Tip
Always inspect schema using:
- printSchema()
- show()

Why?
Because wrong data types will silently break your logic later.


## Filtering Data

    show salaries greater than 4000 and age > 23

In [165]:
df.select("salary").filter("salary > 4000 and age > 23")

salary
7000


In [167]:
df.select("salary").filter( (df["salary"] > 4000) & (df["age"] > 23) )

salary
7000


## Renaming Columns

rename name column to be full_name

In [171]:
df_renamed = df.withColumnRenamed("name" , "full_name")
df_renamed.show()

+----+---+---------+------+
| age| id|full_name|salary|
+----+---+---------+------+
|  25|  1|      Ali|  4000|
|  30|  2|     Sara|  7000|
|  22|  3|     Omar|  null|
|null|  4|     Mona|  5000|
+----+---+---------+------+



## Reading CSV with proper schema

In [206]:
customers_schema = """ id INTEGER, name STRING, city STRING, address STRING """

customers_df = spark.read.csv("file:///data/customers.csv", schema = customers_schema, mode= 'DROPMALFORMED')
customers_df.show()
customers_df.printSchema()

+---+--------+-----+--------------------+
| id|    name| city|             address|
+---+--------+-----+--------------------+
|  1|     Ali|Sanaa|           Street 10|
|  2|Mohammed| Aden|          Building 5|
|  3|    Omar|Sanaa|              Zone 3|
|  4|   Ahmed|  Ibb|Old Street Behind...|
|  5|  Hassan|Sanaa|       Al-Zubairi St|
+---+--------+-----+--------------------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- address: string (nullable = true)




## 🧪 Data Quality Check

Find:
- Rows with null salary
- Rows with null age


In [189]:
df.filter(df["salary"].isNull())


age,id,name,salary
22,3,Omar,null


In [190]:
df.filter("age is null")

age,id,name,salary
null,4,Mona,5000


Handle null values:
- Replace null salary with 0
- Drop rows where age is null

In [191]:
df_fill_null = df.na.fill({"salary":0})
df_fill_null.show()
df_fill_null.printSchema()

+----+---+----+------+
| age| id|name|salary|
+----+---+----+------+
|  25|  1| Ali|  4000|
|  30|  2|Sara|  7000|
|  22|  3|Omar|     0|
|null|  4|Mona|  5000|
+----+---+----+------+

root
 |-- age: long (nullable = true)
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = false)



In [192]:
df_drop_null = df_fill_null.na.drop(subset=["age"])
df_drop_null.show()

+---+---+----+------+
|age| id|name|salary|
+---+---+----+------+
| 25|  1| Ali|  4000|
| 30|  2|Sara|  7000|
| 22|  3|Omar|     0|
+---+---+----+------+

